<a href="https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule flags a page for refresh when it is stale (not updated in 180+ days), still visible (above-median impressions), and actively declining (impressions_last30 < impressions_prev30). Pages meeting all three get a score weighted by impressions.

Reason codes:

* stale_visible_declining — all three conditions met, highest priority
* stale_declining — stale and declining but low impressions, lower priority
* hold — does not meet threshold, no action needed

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): ")

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')")

In [5]:
REL = "hf://datasets/FlyRank/internship-warehouse"

In [7]:
TABLES = {
    "dim_clients":  f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":  f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [13]:
df = con.execute(f"""
    WITH base AS (
        SELECT
            d.content_hash_id,
            d.client_hash_id,
            d.month,
            d.report_date,
            d.gsc_impressions,
            d.gsc_clicks,
            d.gsc_avg_position,
            MAX(d.report_date) OVER (
                PARTITION BY d.content_hash_id, d.client_hash_id
            ) AS max_date
        FROM {TABLES['fact_daily']} d
        WHERE d.month = '2026-03'
          AND d.gsc_data_available = TRUE
    )
    SELECT
        b.content_hash_id,
        b.client_hash_id,
        b.month,
        SUM(b.gsc_impressions)                                       AS impressions_90d,
        SUM(b.gsc_clicks)                                            AS clicks_90d,
        AVG(b.gsc_avg_position)                                      AS avg_position,
        SUM(b.gsc_clicks) / NULLIF(SUM(b.gsc_impressions), 0)       AS ctr,
        SUM(CASE WHEN b.report_date >= (b.max_date - INTERVAL 30 DAYS)
                 THEN b.gsc_impressions ELSE 0 END)                  AS impressions_last30,
        SUM(CASE WHEN b.report_date < (b.max_date - INTERVAL 30 DAYS)
                 THEN b.gsc_impressions ELSE 0 END)                  AS impressions_prev30,
        c.content_type,
        c.word_count,
        c.last_optimized_date,
        DATEDIFF('day', c.last_optimized_date, MAX(b.report_date))  AS days_since_last_update,
        DATEDIFF('day', c.content_created_date, MAX(b.report_date)) AS content_age_days
    FROM base b
    JOIN {TABLES['dim_content']} c
        ON b.content_hash_id = c.content_hash_id
    GROUP BY
        b.content_hash_id, b.client_hash_id, b.month,
        c.content_type, c.word_count,
        c.last_optimized_date, c.content_created_date
""").df()

# Derive trend_direction
df["trend_direction"] = np.where(
    df["impressions_last30"] > df["impressions_prev30"], "up",
    np.where(df["impressions_last30"] < df["impressions_prev30"], "down", "flat")
)

print(f"Rows loaded: {len(df):,}")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 176,738


,content_hash_id,client_hash_id,month,impressions_90d,clicks_90d,avg_position,ctr,impressions_last30,impressions_prev30,content_type,word_count,last_optimized_date,days_since_last_update,content_age_days,trend_direction
0,content_14a3d47ccd0d15dc,client_2094c6eb080311d5,2026-03,2.0,0.0,5.500000,0.0,2.0,0.0,keyword article,3864,NaT,<NA>,90,up
1,content_14a6f92117604fef,client_2094c6eb080311d5,2026-03,90.0,0.0,7.936301,0.0,90.0,0.0,keyword article,2867,2026-05-20,-50,110,up
2,content_14ef58c38dd0ff6f,client_2094c6eb080311d5,2026-03,8.0,0.0,33.777778,0.0,8.0,0.0,keyword article,2928,NaT,<NA>,49,up
3,content_1515e6f85acc54e1,client_2094c6eb080311d5,2026-03,11.0,0.0,15.366667,0.0,11.0,0.0,keyword article,3140,NaT,<NA>,36,up
4,content_156eaf2e40815dac,client_2094c6eb080311d5,2026-03,2.0,0.0,4.000000,0.0,2.0,0.0,keyword article,5115,NaT,<NA>,82,up


In [14]:
# Define signals
stale     = df["days_since_last_update"] >= 180
visible   = df["impressions_90d"] >= df["impressions_90d"].median()
declining = df["trend_direction"] == "down"

print(f"Stale pages:             {stale.sum():,}")
print(f"Visible pages:           {visible.sum():,}")
print(f"Declining pages:         {declining.sum():,}")
print(f"All three (top targets): {(stale & visible & declining).sum():,}")

Stale pages:             0
Visible pages:           88,478
Declining pages:         0
All three (top targets): 0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = impressions_90d × stale × declining. Visibility gates entry; impressions break ties. Every row gets a reason code and action label before writing to CSV.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
# Score
df["baseline_score"] = np.where(
    stale & visible & declining,
    df["impressions_90d"],
    np.where(stale & declining, df["impressions_90d"] * 0.5, 0)
)

# Reason code
def assign_reason(row):
    if row["baseline_score"] == 0:
        return "hold"
    elif row["impressions_90d"] >= df["impressions_90d"].median():
        return "stale_visible_declining"
    else:
        return "stale_declining"

df["reason_code"]  = df.apply(assign_reason, axis=1)
df["action_label"] = np.where(df["baseline_score"] > 0, "refresh", "hold")

# Ranked queue
queue = df.sort_values("baseline_score", ascending=False)

# Write CSV
os.makedirs("work/outputs", exist_ok=True)
queue[["content_hash_id", "client_hash_id", "month", "baseline_score",
       "reason_code", "action_label", "impressions_90d",
       "days_since_last_update", "ctr", "avg_position",
       "trend_direction"]]\
    .to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Total rows:      {len(queue):,}")
print(f"Flagged refresh: {(queue['action_label']=='refresh').sum():,}")
print(f"Hold:            {(queue['action_label']=='hold').sum():,}")
queue[["content_hash_id", "baseline_score", "reason_code", "action_label"]].head()

Total rows:      176,738
Flagged refresh: 0
Hold:            176,738


,content_hash_id,baseline_score,reason_code,action_label
176737,content_e79ef35064aaad1b,0.0,hold,hold
0,content_14a3d47ccd0d15dc,0.0,hold,hold
176721,content_96115909d97214fd,0.0,hold,hold
176720,content_960b8944ae786ce1,0.0,hold,hold
176719,content_96028eaa141dddfd,0.0,hold,hold


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [16]:
top20 = queue.head(20)[["content_hash_id", "client_hash_id", "baseline_score",
                          "reason_code", "action_label", "impressions_90d",
                          "days_since_last_update", "ctr",
                          "avg_position", "trend_direction"]]
print(top20.to_string())

                 content_hash_id           client_hash_id  baseline_score reason_code action_label  impressions_90d  days_since_last_update       ctr  avg_position trend_direction
176737  content_e79ef35064aaad1b  client_810019792c9b8efc             0.0        hold         hold             50.0                    <NA>  0.000000      9.479770              up
0       content_14a3d47ccd0d15dc  client_2094c6eb080311d5             0.0        hold         hold              2.0                    <NA>  0.000000      5.500000              up
176721  content_96115909d97214fd  client_fef1a8f436438636             0.0        hold         hold           1126.0                    <NA>  0.000888      5.768734              up
176720  content_960b8944ae786ce1  client_fef1a8f436438636             0.0        hold         hold            176.0                     -51  0.000000     12.366257              up
176719  content_96028eaa141dddfd  client_fef1a8f436438636             0.0        hold         hold  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Pages near the top with avg_position > 8 concern me — a stale declining page that Google barely ranks may have a relevance problem a content refresh cannot fix. Leakage check below confirms no future-window columns used.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [18]:
# Leakage check — all inputs are knowable at decision time
numeric_inputs = ["days_since_last_update", "impressions_90d",
                  "impressions_last30", "impressions_prev30"]
text_inputs    = ["trend_direction"]

print("=== Leakage check ===")
for col in numeric_inputs:
    print(f"{col}: min={df[col].min():.1f}, max={df[col].max():.1f}, nulls={df[col].isnull().sum()}")

for col in text_inputs:
    print(f"{col}: values={df[col].unique().tolist()}, nulls={df[col].isnull().sum()}")

# Weak picks
weak = queue.head(20)[queue.head(20)["avg_position"] > 8]
print(f"\nWeak picks in top 20 (avg_position > 8): {len(weak)}")
print(weak[["content_hash_id", "baseline_score",
            "avg_position", "reason_code"]].to_string())

print("\nNo future-window columns used")
print("No label-derived inputs used")
print("trend_direction derived from historical impressions only")

=== Leakage check ===
days_since_last_update: min=-125.0, max=-24.0, nulls=136974
impressions_90d: min=1.0, max=617124.0, nulls=0
impressions_last30: min=1.0, max=617124.0, nulls=0
impressions_prev30: min=0.0, max=0.0, nulls=0
trend_direction: values=['up'], nulls=0

Weak picks in top 20 (avg_position > 8): 13
                 content_hash_id  baseline_score  avg_position reason_code
176737  content_e79ef35064aaad1b             0.0      9.479770        hold
176720  content_960b8944ae786ce1             0.0     12.366257        hold
176719  content_96028eaa141dddfd             0.0     17.491458        hold
176718  content_95ff8a8521d6a023             0.0     17.344789        hold
176717  content_95e76b010edc1e46             0.0     27.858948        hold
176715  content_95cfc181916628f2             0.0     15.118842        hold
176712  content_959786f0fec354c9             0.0     12.250000        hold
176711  content_957d75f073572c06             0.0     15.580134        hold
176710  conte

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.